# MINOR PROJECT 1


In [352]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3146 entries, 0 to 3145
Data columns (total 4 columns):
 #   Column                                                                             Non-Null Count  Dtype 
---  ------                                                                             --------------  ----- 
 0   01/04/24                                                                           3146 non-null   object
 1    01:12 - Messages and calls are end-to-end encrypted. No one outside of this chat  3146 non-null   object
 2    not even WhatsApp                                                                 387 non-null    object
 3    can read or listen to them. Tap to learn more.                                    130 non-null    object
dtypes: object(4)
memory usage: 98.4+ KB


In [396]:
import datetime as dt
import numpy as np


# 1. THE CHAT PARSER (FEATURE 1)

def parse_whatsapp_chat(file_path):
    system_messages_count = 0

    media_omitted_count = {}
    deleted_messages_count = {}
    valid_parsed_messages = []

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            all_lines = f.read().split('\n')
    except Exception:
        print("Error: Could not open file path! Check if it is uploaded.")
        return [], [], 0, {}, {}

    for line in all_lines:
        line = line.strip()
        if not line:
            continue

        if len(line) >= 8 and line[2] == '/' and line[5] == '/':
            if ' - ' in line:
                parts = line.split(' - ', 1)
            else:

                parts = line.split(' ', 1)

            timestamp_str = parts[0]
            remaining_text = parts[1]

            if ':' not in remaining_text:
                system_messages_count += 1
                continue


            sender_name, message_body = remaining_text.split(':', 1)
            sender_name = sender_name.strip()
            message_body = message_body.strip()

            if message_body == "<Media omitted>":
                if sender_name not in media_omitted_count:
                    media_omitted_count[sender_name] = 0
                media_omitted_count[sender_name] += 1
                message_dict = {
                    'timestamp': timestamp_str,
                    'sender': sender_name,
                    'text': message_body,
                    'is_media': True,
                    'is_deleted': False
                }
                valid_parsed_messages.append(message_dict)
                continue

            if message_body == "This message was deleted":
                if sender_name not in deleted_messages_count:
                    deleted_messages_count[sender_name] = 0
                deleted_messages_count[sender_name] += 1

                message_dict = {
                    'timestamp': timestamp_str,
                    'sender': sender_name,
                    'text': message_body,
                    'is_media': False,
                    'is_deleted': True
                }
                valid_parsed_messages.append(message_dict)
                continue

            # Normal message handling
            message_dict = {
                'timestamp': timestamp_str,
                'sender': sender_name,
                'text': message_body,
                'is_media': False,
                'is_deleted': False
            }
            valid_parsed_messages.append(message_dict)
        else:
            if len(valid_parsed_messages) > 0:
                last_message_index = len(valid_parsed_messages) - 1
                valid_parsed_messages[last_message_index]['text'] += " " + line

    unique_members = []
    for msg in valid_parsed_messages:
        current_sender = msg['sender']
        if current_sender not in unique_members:
            unique_members.append(current_sender)
    print("=======================================================")
    print("                  *THE CHAT PARSER* ")
    print("=======================================================")
    print("Successfully parsed " + str(len(valid_parsed_messages)) + " messages from " + str(len(unique_members)) + " participants.")
    print("Skipped " + str(system_messages_count) + " system messages.")

    return valid_parsed_messages, unique_members, system_messages_count, media_omitted_count, deleted_messages_count

parsed_data, group_members, system_msgs, media_stats, deleted_stats = parse_whatsapp_chat('/content/hostel_bois.txt')


def get_count_value(item):
    return item[1]

def generate_group_overview(messages, members):
    total_msg = len(messages)
    first_date = messages[0]['timestamp'].split(',')[0].strip()
    last_date = messages[total_msg - 1]['timestamp'].split(',')[0].strip()

    person_counts = {}
    for person in members:
        person_counts[person] = 0

    for msg in messages:
        sender = msg['sender']
        person_counts[sender] += 1

    sorted_stats = sorted(person_counts.items(), key=get_count_value, reverse=True)

    print("=======================================================")
    print("                  *GROUP OVERVIEW*")
    print("=======================================================")
    print("Group            : Hostel Bois 4ever")
    print("Period           : " + first_date + " to " + last_date + " (60 days)")
    print("Total messages   : " + str(total_msg))
    print("Participants     : " + str(len(members)))

    print("=======================================================")
    print("                 *MESSAGES PER PERSON*")
    print("=======================================================")

    for member, count in sorted_stats:
        percentage_share = (count / total_msg) * 100
        print(f"{member:<10} : {count:>4} ({percentage_share:.1f}%)")
    return person_counts

member_activity_counts = generate_group_overview(parsed_data, group_members)

def find_busiest_activity_windows(messages):

    days_tracker = {}
    hours_tracker = {}

    for msg in messages:
        time_segments = msg['timestamp'].split(', ')
        date_key = time_segments[0].strip()              # Extracts the date (e.g., "14/04/24")
        hour_key = time_segments[1].split(':')[0].strip() # Extracts the hour (e.g., "17")

        # counts for dates
        if date_key not in days_tracker:
            days_tracker[date_key] = 0
        days_tracker[date_key] = days_tracker[date_key] + 1

        # counts for hours
        if hour_key not in hours_tracker:
            hours_tracker[hour_key] = 0
        hours_tracker[hour_key] = hours_tracker[hour_key] + 1


    top_day = max(days_tracker.items(), key=get_count_value)
    top_hour = max(hours_tracker.items(), key=get_count_value)


    print("Busiest day      : " + str(top_day[0]) + " (" + str(top_day[1]) + " messages)")
    print("Busiest hour     : " + str(top_hour[0]) + ":00 (" + str(top_hour[1]) + " total messages)")

    return top_day, top_hour
peak_day, peak_hour = find_busiest_activity_windows(parsed_data)

## Feature 4: Activity Heatmap (NumPy)

def run_numpy_heatmap(messages, members):
  heatmap_data = np.zeros((len(members), 24),dtype=int)
  lookup_indices = {}
  for index in range(len(members)):
    name = members[index]
    lookup_indices[members[index]] = index

  for msg in messages:
    sender = msg['sender']
    try:
      hour_val = int(msg['timestamp'].split(', ')[1].split(':')[0])
      row = lookup_indices[sender]
      heatmap_data[row, hour_val] = heatmap_data[row, hour_val] + 1
    except Exception:
      continue # Corrected: Indented 'continue' under 'except'

  print("=========================================================")
  print("           *ACTIVITY HEATMAP (messages by hour)*")
  print("=========================================================")
  print(" " * 12 + "00 03 06 09 12 15 18 21")

  for idx in range(len(members)):
        member = members[idx]
        display_line = f"{member:<12}"

        row_max = np.max(heatmap_data[idx, :])
        if row_max == 0:
          row_max = 1 # Avoid division by zero

        for target_hour in [0, 3, 6, 9, 12, 15, 18, 21]:
            current_val = heatmap_data[idx, target_hour]
            ratio = current_val / row_max
            if ratio < 0.25:
                display_line = display_line + " . "  # 0 - 25% activity
            elif ratio < 0.50:
                display_line = display_line + " ░ "  # 25 - 50% activity
            elif ratio < 0.75:
                display_line = display_line + " ▒ "  # 50 - 75% activity
            else:
                display_line = display_line + " █ "  # 75 - 100% activity

        print(display_line)
  return heatmap_data

heatmap_matrix = run_numpy_heatmap(parsed_data, group_members)


## Feature 5: Top Words with Block Character Bar Visualization

def generate_favourite_words(messages):
    filter_words = ['i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'you', 'me', 'it', 'my', 'that', 'with', 'was', 'how', 'this', 'so', 'about', 'at', 'am', 'how','hai','he','his','how','just','which','have','telling','from','up','one','we','are','had','started','they'] # Removed specific words as requested
    word_frequencies = {}


    for msg in messages:

        if msg['is_media'] or msg['is_deleted']:
            continue

        words_list = msg['text'].lower().split()
        for word in words_list:
            cleaned = word.strip(".,?!()[]{}*_-:;\"'")

            if cleaned and cleaned not in filter_words:
                word_frequencies[cleaned] = word_frequencies.get(cleaned, 0) + 1 # Corrected frequency counting logic


    top_words = sorted(word_frequencies.items(), key=get_count_value, reverse=True)[:5]

    print("=======================================================")
    print("             *THIS GROUP'S FAVOURITE WORDS*")
    print("=======================================================")


    max_count = 1
    if len(top_words) > 0:
        max_count = top_words[0][1]

    for word, count in top_words:

        bar_scale = int((count / max_count) * 15)
        bar_graphic = "█" * bar_scale


# {word:<12} left-aligns the word token within 12 character slots
# {count:>4} right-aligns the count numbers within 4 character slots
        print(f"{word:<12} {count:>4}  {bar_graphic}")

    return top_words

top_five_words = generate_favourite_words(parsed_data)


## Feature 6: Response Speed & Silent Streaks

def analyze_timings_and_streaks(messages, members):

    user_gaps = {}
    for m in members:
        user_gaps[m] = []

    for i in range(1, len(messages)):
        last_msg = messages[i-1]
        this_msg = messages[i]

        if last_msg['sender'] != this_msg['sender']:
            try:
                format_str = '%d/%m/%y, %H:%M'
                time1 = dt.datetime.strptime(last_msg['timestamp'].strip(), format_str)
                time2 = dt.datetime.strptime(this_msg['timestamp'].strip(), format_str)
                delta_minutes = (time2-time1).total_seconds() / 60.0

                if delta_minutes >= 0 and delta_minutes <= 360:
                    user_gaps[this_msg['sender']].append(delta_minutes)
            except Exception:
                continue

    print("=======================================================")
    print("                *RESPONSE PATTERNS  *")
    print("=======================================================")

    average_speeds = {}
    for member in members:
        gaps = user_gaps[member]
        # Calculate the simple arithmetic mean of the user's recorded minutes list
        if len(gaps) > 0:
            avg = sum(gaps) / len(gaps)
        else:
            avg = 999.0 # Fallback high number if a user never replied
        average_speeds[member] = avg

    # Sort the average speeds to find the fastest and slowest repliers
    fastest_replier = sorted(average_speeds.items(), key=get_count_value, reverse=False)[0]
    slowest_replier = sorted(average_speeds.items(), key=get_count_value, reverse=True)[0]

    print("Fastest replier  : " + fastest_replier[0] + " (avg " + f"{fastest_replier[1]:.1f}" + " minutes)")
    print("Slowest replier  : " + slowest_replier[0] + " (avg " + f"{slowest_replier[1]:.1f}" + (" hours)" if slowest_replier[1] > 60 else " minutes)"))
    print("=======================================================")
    print("=================================================================")
    print("*LONGEST SILENT STREAKS (consecutive days with zero messages)*")
    print("=================================================================")

# 1. build a complete 60-day chronological calendar timeline
    calendar_timeline = []
    base_date = dt.datetime.strptime(messages[0]['timestamp'].split(',')[0], '%d/%m/%y')

    for offset in range(60):
        current_step = base_date + dt.timedelta(days=offset)
        calendar_timeline.append(current_step.strftime('%d/%m/%y'))

# 2. Check each member's consecutive absent days across the timeline
    for member in members:
        active_dates_pool = []
  # Gather all calendar dates this person actually spoke on
        for msg in messages:
            if msg['sender'] == member:
                active_dates_pool.append(msg['timestamp'].split(',')[0].strip())

        current_run = 0
        longest_run = 0
        longest_streak_start_date = None
        longest_streak_end_date = None


        for date_entry in calendar_timeline:
            if date_entry not in active_dates_pool:
                if current_run == 0:
                    current_streak_start_date = date_entry
                current_run += 1
                if current_run > longest_run:
                    longest_run = current_run
                    longest_streak_start_date = current_streak_start_date
                    longest_streak_end_date = date_entry
            else:
                current_run = 0

        streak_info = f"{member:<10} : {longest_run} days"
        if longest_run > 0 and longest_streak_start_date and longest_streak_end_date:
            streak_info += f" ({longest_streak_start_date} to {longest_streak_end_date})"
        elif longest_run == 0:
            streak_info += " (never went silent)"
        print(streak_info)

analyze_timings_and_streaks(parsed_data, group_members)


## Feature 7: Personality Archetype Detection


def calculate_archetype_profiles(messages, members, heatmap_data):

    user_indices = {}
    for index in range(len(members)):
        user_indices[members[index]] = index

    last_tracked_sender = None
    assigned_roles = {}

    burst_records = {}
    for m in members:
        burst_records[m] = []

    accumulated_burst = 0


    for msg in messages:
        if last_tracked_sender != msg['sender']:
            last_tracked_sender = msg['sender']
            accumulated_burst = 1
        elif msg['sender'] == last_tracked_sender:
            accumulated_burst = accumulated_burst + 1
        else:
            # Save the message burst count for the previous sender
            burst_records[last_tracked_sender].append(accumulated_burst)
            last_tracked_sender = msg['sender']
            accumulated_burst = 1

    # List of caring keywords defined in the project brief for Group Mom tracking
    mom_lexicon = ['okay', 'safe', 'eat', 'sleep', 'take care', 'please', 'reminder']

    # EVALUATE METRICS PER MEMBER
    for member in members:
        profile_scores = {}
        row = user_indices[member]

# 1. Rule Evaluation: Spammer Score
        member_bursts = burst_records[member]
        if len(member_bursts) > 0:
            avg_burst = sum(member_bursts) / len(member_bursts)
        else:
            avg_burst = 1.0
        profile_scores['THE SPAMMER'] = avg_burst / 5.0

# 2. Rule Evaluation: Night Owl Score
        after_dark_volume = np.sum(heatmap_data[row, [23, 0, 1, 2, 3, 4]])
        total_volume = np.sum(heatmap_data[row, :])
        if total_volume > 0:
            profile_scores['THE NIGHT OWL'] = after_dark_volume / total_volume
        else:
            profile_scores['THE NIGHT OWL'] = 0


        text_lengths = []
        caring_hits = 0
        all_caps_hits = 0
        valid_evaluations = 0

        for msg in messages:
            if msg['sender'] == member and msg['is_media'] == False and msg['is_deleted'] == False:
                valid_evaluations = valid_evaluations + 1
                body = msg['text']
                tokens = body.split()
                text_lengths.append(len(tokens))

                # Check for Group Mom caring keywords
                for token in tokens:
                    if token.lower().strip(".,?!") in mom_lexicon:
                        caring_hits = caring_hits + 1

                # Check for Drama Queen constraints
                if len(body) > 3 and body.isupper():
                    all_caps_hits = all_caps_hits + 1

# 3. Rule Evaluation: Storyteller Score
        if len(text_lengths) > 0:
            avg_word_count = sum(text_lengths) / len(text_lengths)
        else:
            avg_word_count = 0
        profile_scores['THE STORYTELLER'] = avg_word_count / 60.0

# 4. Rule Evaluation: Group Mom Score
        profile_scores['THE GROUP MOM'] = caring_hits / 10.0

# 5. Rule Evaluation: Drama Queen Score
        if valid_evaluations > 0:
            profile_scores['THE DRAMA QUEEN'] = (all_caps_hits / valid_evaluations)
        else:
            profile_scores['THE DRAMA QUEEN'] = 0

# 6. Rule Evaluation: Ghost Score (Using np.count_nonzero on active hours)
        active_hours_count = np.count_nonzero(heatmap_data[row, :])
        profile_scores['THE GHOST'] = 1.0 - (active_hours_count / 24.0)

# EXCLUSIVE ASSIGNMENT RESOLUTION

        best_archetype = sorted(profile_scores.items(), key=get_count_value, reverse=True)[0][0]
        assigned_roles[member] = best_archetype

    print("=======================================================")
    print("               *PERSONALITY ARCHETYPES*  ")
    print("=======================================================")
    for name, role in assigned_roles.items():
        print(f"{name:<10} → {role}")

    return assigned_roles
archetype_results = calculate_archetype_profiles(parsed_data, group_members, heatmap_matrix)
print("\n FINAL PERSONALITY CLASSIFICATION")
for name, role in archetype_results.items():
    print(f"  ▪ {name:<14} ➔  {role}")

print("============================================================")
print("         Generated Safely via GroupDNA Engine 2026          ")
print("============================================================")

                  *THE CHAT PARSER* 
Successfully parsed 3174 messages from 6 participants.
Skipped 4 system messages.
                  *GROUP OVERVIEW*
Group            : Hostel Bois 4ever
Period           : 01/04/24 to 30/05/24 (60 days)
Total messages   : 3174
Participants     : 6
                 *MESSAGES PER PERSON*
Rahul      :  953 (30.0%)
Priya      :  718 (22.6%)
Neha       :  635 (20.0%)
Aman       :  490 (15.4%)
Karan      :  354 (11.2%)
Vikas      :   24 (0.8%)
Busiest day      : 04/05/24 (76 messages)
Busiest hour     : 18:00 (248 total messages)
           *ACTIVITY HEATMAP (messages by hour)*
            00 03 06 09 12 15 18 21
Rahul        .  .  .  .  ▒  ▒  █  █ 
Priya        .  .  .  █  █  ░  ▒  ░ 
Karan        .  .  .  ░  █  ▒  ▒  ░ 
Neha         .  .  .  █  ▒  .  █  ░ 
Aman         ▒  ▒  .  .  .  .  .  . 
Vikas        .  .  .  ░  ▒  ░  ▒  ░ 
             *THIS GROUP'S FAVOURITE WORDS*
guys          318  ███████████████
today         257  ████████████
everyone      